<a href="https://colab.research.google.com/github/coderhema/adtc-2026-submission-template/blob/main/adtc_qlora_benchmark_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hekima - ADTC 2026 Education Fine-Tune Benchmark (base vs QLoRA)

**Question:** does fine-tuning tiny-aya-global on math and scientific-reasoning data (AfriGSM math + science past questions + bilingual step-by-step pairs) improve (a) arc_easy - the automated half of S_acc - and (b) bilingual reasoning-style answering, the judge half?

**Flow:** base evals (arc_easy 50-shot-5 + 20-question WAEC set EN/YO/HA/SW/IG + the 2 exact metadata.json judge prompts) -> QLoRA fine-tune on T4 -> identical evals -> comparison charts + verdict JSON. Total runtime ~40-90 min on free Colab T4.

Run: Runtime -> Run all. GPU needed (T4 is fine).

## Setup
1. **HF access (one-time):** open https://huggingface.co/CohereLabs/tiny-aya-global, click *Agree and access repository* (instant).
2. **HF token (one-time):** https://huggingface.co/settings/tokens -> create a *read* token.
3. Run Cell 1 (install), Cell 2 (login, paste token), then continue.

In [ ]:
# Cell 1: install dependencies (run once)
# tiny-aya-global needs a recent transformers (cohere2 arch support).
# torchao >= 0.16.0 is required by current peft versions.
!pip install -q -U "transformers>=4.46" "peft>=0.14" "trl>=0.13,<0.16" "torchao>=0.16.0" bitsandbytes accelerate datasets matplotlib seaborn

# Check GPU (free Colab gives T4, ~15 GB VRAM - plenty for a 3.3B 4-bit model)
!nvidia-smi

In [ ]:
# Cell 2: Hugging Face login (REQUIRED once)
# The base model CohereLabs/tiny-aya-global is access-restricted.
# 1. Open https://huggingface.co/CohereLabs/tiny-aya-global and click "Agree and access repository".
# 2. Create a read token: https://huggingface.co/settings/tokens
# 3. Set it as the HF_TOKEN Colab secret, or paste it when prompted below.
import os
from huggingface_hub import login, notebook_login

_TOKEN = os.environ.get("HF_TOKEN", "")
if _TOKEN.startswith("hf_"):
    login(token=_TOKEN)
    print("HF login ok (from HF_TOKEN)")
else:
    notebook_login()  # interactive fallback


In [ ]:

# Cell 3: configuration
import json, os, re, random, gc
import numpy as np
import torch
from datasets import load_dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, TrainingArguments)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_ID = "CohereLabs/tiny-aya-global"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# arc_easy eval size (the ADTC profiler uses 50 by default; real audit uses more)
ARC_N = 50
ARC_SHOT = 5
MAX_NEW = 128

OUT = "/content/adtc_results"
os.makedirs(OUT, exist_ok=True)
print("config ok")

In [ ]:

# Cell 4: load base model in 4-bit (QLoRA-ready)
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                                 device_map="auto", dtype=torch.float16)
except Exception as e:
    print("First load failed (", type(e).__name__, "), retrying with trust_remote_code=True")
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                                 device_map="auto", dtype=torch.float16,
                                                 trust_remote_code=True)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
print("loaded:", MODEL_ID)
print("params trainable-check: model is 4-bit base, ready for LoRA")

## Baseline (before fine-tuning)

In [ ]:
# Cell 5: evaluation helpers (arc_easy + WAEC bilingual set)
def build_arc_prompt(q, choices, shots):
    lines = ["The following are multiple choice questions (with answers) about science."]
    for s in shots:
        sq, sch, sans = s["question"], s["choices"], s["answer"]
        lines.append("")
        lines.append(f"Question: {sq}")
        lines.append("Choices:")
        # Use actual keys from the dictionary instead of hardcoded "ABCD"
        for L in sorted(sch.keys()):
            lines.append(f"{L}. {sch[L]}")
        lines.append(f"Answer: {sans}")
    lines.append("")
    lines.append(f"Question: {q}")
    lines.append("Choices:")
    for L in sorted(choices.keys()):
        lines.append(f"{L}. {choices[L]}")
    lines.append("Answer:")
    return "\n".join(lines)

def letters_loglik(model, tok, prompt, labels):
    """Score each valid letter/label appended after the prompt."""
    base_ids = tok(prompt, return_tensors="pt").input_ids.to(model.device)
    probs = {}
    with torch.no_grad():
        for L in labels:
            ids = torch.cat([base_ids, tok(" " + str(L), return_tensors="pt").input_ids.to(model.device)], dim=1)
            out = model(ids).logits[0, -2:-1, :]
            logp = torch.log_softmax(out, dim=-1)[0, ids[0, -1]].item()
            probs[L] = logp
    return max(probs, key=probs.get), probs

def generate_answer(model, tok, prompt, max_new=MAX_NEW):
    ids = tok(prompt, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

def extract_letter(text, labels):
    if not text: return None
    pattern = r"\b(" + "|".join(map(re.escape, map(str, labels))) + r")\b"
    m = re.search(pattern, text)
    return m.group(1) if m else None

def eval_arc(model, tok, train_rows, test_rows, n=ARC_N, shots=ARC_SHOT):
    test_rows = test_rows[:n]
    ll_correct = 0; gen_correct = 0
    for i, row in enumerate(test_rows):
        prompt = build_arc_prompt(row["question"], row["choices"], train_rows[:shots])
        valid_labels = list(row["choices"].keys())
        letter_ll, _ = letters_loglik(model, tok, prompt, valid_labels)
        if str(letter_ll) == str(row["answer"]): ll_correct += 1
        gen = generate_answer(model, tok, prompt)
        letter_gen = extract_letter(gen, valid_labels)
        if str(letter_gen) == str(row["answer"]): gen_correct += 1
        if (i + 1) % 10 == 0:
            print(f"  arc_easy {i+1}/{n}  ll_acc={ll_correct/(i+1):.3f}  gen_acc={gen_correct/(i+1):.3f}")
    return {"arc_easy_ll_acc": ll_correct / n, "arc_easy_gen_acc": gen_correct / n, "n": n}

def chat_prompt(question, choices=None):
    if choices:
        q = question + "\n\nChoices:\n" + "\n".join(f"{L}. {choices[L]}" for L in sorted(choices.keys()))
    else:
        q = question
    return tok.apply_chat_template([{"role": "user", "content": q}],
                                   tokenize=False, add_generation_prompt=True)

def eval_waec(model, tok, items):
    results = []
    for it in items:
        prompt = chat_prompt(it["question"], it["choices"])
        valid_labels = list(it["choices"].keys())
        gen = generate_answer(model, tok, prompt)
        letter = extract_letter(gen, valid_labels)
        correct = str(letter) == str(it["answer"])
        results.append({**it, "generated": gen, "letter": letter, "correct": correct})
    return results

def waec_summary(results):
    langs = {}
    if not results: return {}, 0.0
    for r in results:
        langs.setdefault(r["lang"], {"n": 0, "c": 0})
        langs[r["lang"]]["n"] += 1
        langs[r["lang"]]["c"] += 1 if r["correct"] else 0
    return {k: round(v["c"] / v["n"], 3) for k, v in langs.items()}, \
           round(sum(1 for r in results if r["correct"]) / len(results), 3)

In [ ]:
import random
# Cell 6: load arc_easy (same source lm-eval uses: allenai/ai2_arc)
arc = load_dataset("allenai/ai2_arc", "ARC-Easy")
def fmt(row):
    q = row["question"]
    # Dynamic mapping to handle rows that might not have exactly 4 choices (A,B,C,D)
    labels = row["choices"]["label"]
    texts = row["choices"]["text"]
    ch = {label: texts[i] for i, label in enumerate(labels)}
    return {"question": q, "choices": ch, "answer": row["answerKey"]}

train_rows = [fmt(r) for r in arc["train"]][:ARC_SHOT]
# Shuffle the training split specifically if needed, though we already took shots
# Note: arc["train"] is a dataset object, we convert to list to shuffle if required,
# but here we just shuffle the row order for future sampling.
test_rows = [fmt(r) for r in arc["test"]]
random.seed(SEED); random.shuffle(test_rows)
print(f"arc_easy loaded: {len(test_rows)} test rows, {len(train_rows)} shots")

In [ ]:
# Cell 7: BASELINE (base model, before fine-tuning) - eval on float16 base
# (matches the precision used for the fine-tuned eval and the exported GGUF)
print("== arc_easy (base) ==")
_base_fp = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
base_arc = eval_arc(_base_fp, tok, train_rows, test_rows, n=ARC_N)
print("baseline arc_easy:", base_arc)
with open(f"{OUT}/base_arc.json", "w") as f: json.dump(base_arc, f, indent=2)
del _base_fp; gc.collect(); torch.cuda.empty_cache()

print("\n== WAEC bilingual set (base) ==")
_base_fp = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
base_waec = eval_waec(_base_fp, tok, WAEC_EVAL)
by_lang, overall = waec_summary(base_waec)
print("by language:", by_lang, "| overall:", overall)
with open(f"{OUT}/base_waec.json", "w") as f:
    json.dump({"by_lang": by_lang, "overall": overall, "items": base_waec}, f, indent=2, ensure_ascii=False)
del _base_fp; gc.collect(); torch.cuda.empty_cache()


In [ ]:

# Cell 8: BASELINE on the bilingual reasoning set (EN/YO/HA/SW/IG)
WAEC_EVAL = json.loads("[{\"lang\": \"en\", \"subject\": \"math\", \"answer\": \"B\", \"question\": \"A trader buys a bag of rice for N24,000 and sells it for N30,000. What is the percentage profit?\", \"choices\": {\"A\": \"20%\", \"B\": \"25%\", \"C\": \"30%\", \"D\": \"15%\"}}, {\"lang\": \"en\", \"subject\": \"math\", \"answer\": \"C\", \"question\": \"If 2x + 5 = 17, what is the value of x?\", \"choices\": {\"A\": \"4\", \"B\": \"5\", \"C\": \"6\", \"D\": \"7\"}}, {\"lang\": \"en\", \"subject\": \"math\", \"answer\": \"B\", \"question\": \"The angles of a triangle are x, 2x and 3x. What is the value of x?\", \"choices\": {\"A\": \"20 degrees\", \"B\": \"30 degrees\", \"C\": \"40 degrees\", \"D\": \"60 degrees\"}}, {\"lang\": \"en\", \"subject\": \"science\", \"answer\": \"B\", \"question\": \"Which of the following is NOT a vector quantity?\", \"choices\": {\"A\": \"Force\", \"B\": \"Mass\", \"C\": \"Velocity\", \"D\": \"Weight\"}}, {\"lang\": \"yo\", \"subject\": \"math\", \"answer\": \"B\", \"question\": \"Oníṣòwò kan ra àpò ìrẹsì kan ní N24,000, ó sì tà á ní N30,000. Kín ni ìpín ọgọ́rùn-ún èrè rẹ̀?\", \"choices\": {\"A\": \"20%\", \"B\": \"25%\", \"C\": \"30%\", \"D\": \"15%\"}}, {\"lang\": \"yo\", \"subject\": \"math\", \"answer\": \"C\", \"question\": \"Bí 2x + 5 = 17, kín ni iye x?\", \"choices\": {\"A\": \"4\", \"B\": \"5\", \"C\": \"6\", \"D\": \"7\"}}, {\"lang\": \"yo\", \"subject\": \"math\", \"answer\": \"B\", \"question\": \"Àwọn igun ìgbá mẹ́ta kan jẹ́ x, 2x àti 3x. Kín ni iye x?\", \"choices\": {\"A\": \"20 ìwọ̀n\", \"B\": \"30 ìwọ̀n\", \"C\": \"40 ìwọ̀n\", \"D\": \"60 ìwọ̀n\"}}, {\"lang\": \"yo\", \"subject\": \"science\", \"answer\": \"B\", \"question\": \"Èwo nínú àwọn wọ̀nyí kì í ṣe òye afẹ̀sọ́nà (vector quantity)?\", \"choices\": {\"A\": \"Agbára\", \"B\": \"Ìwúwo\", \"C\": \"Ìyára\", \"D\": \"Àdánwó\"}}, {\"lang\": \"ha\", \"subject\": \"math\", \"answer\": \"B\", \"question\": \"Wani ɗan kasuwa ya sayi buhun shinkafa akan N24,000 ya kuma sayar da shi akan N30,000. Menene kashi na riba?\", \"choices\": {\"A\": \"20%\", \"B\": \"25%\", \"C\": \"30%\", \"D\": \"15%\"}}, {\"lang\": \"ha\", \"subject\": \"math\", \"answer\": \"C\", \"question\": \"Idan 2x + 5 = 17, menene darajar x?\", \"choices\": {\"A\": \"4\", \"B\": \"5\", \"C\": \"6\", \"D\": \"7\"}}, {\"lang\": \"ha\", \"subject\": \"math\", \"answer\": \"B\", \"question\": \"Kusurwoyin alwatika sune x, 2x da 3x. Menene darajar x?\", \"choices\": {\"A\": \"20 digiri\", \"B\": \"30 digiri\", \"C\": \"40 digiri\", \"D\": \"60 digiri\"}}, {\"lang\": \"ha\", \"subject\": \"science\", \"answer\": \"B\", \"question\": \"Wanne daga cikin waɗannan ba shi ne vector ba?\", \"choices\": {\"A\": \"Ƙarfi\", \"B\": \"Nauyi (mass)\", \"C\": \"Gudu\", \"D\": \"Nauyin da aka auna (weight)\"}}, {\"lang\": \"sw\", \"subject\": \"math\", \"answer\": \"B\", \"question\": \"Mfanyabiashara alinunua gunia la mchele kwa N24,000 na kauza kwa N30,000. Ni asilimia ngapi ya faida?\", \"choices\": {\"A\": \"20%\", \"B\": \"25%\", \"C\": \"30%\", \"D\": \"15%\"}}, {\"lang\": \"sw\", \"subject\": \"math\", \"answer\": \"C\", \"question\": \"Ikiwa 2x + 5 = 17, thamani ya x ni?\", \"choices\": {\"A\": \"4\", \"B\": \"5\", \"C\": \"6\", \"D\": \"7\"}}, {\"lang\": \"sw\", \"subject\": \"math\", \"answer\": \"B\", \"question\": \"Pembe za pembetatu ni x, 2x na 3x. Thamani ya x ni?\", \"choices\": {\"A\": \"nyuzi 20\", \"B\": \"nyuzi 30\", \"C\": \"nyuzi 40\", \"D\": \"nyuzi 60\"}}, {\"lang\": \"sw\", \"subject\": \"science\", \"answer\": \"B\", \"question\": \"Ni ipi kati ya hizi ambayo SI vector?\", \"choices\": {\"A\": \"Nguvu\", \"B\": \"Uzito (mass)\", \"C\": \"Kasi\", \"D\": \"Uzito (weight)\"}}, {\"lang\": \"ig\", \"subject\": \"math\", \"answer\": \"B\", \"question\": \"Onye ahịa zụtara akpa osikapa na N24,000 wee ree ya na N30,000. Gịnị bụ pasentị uru ọ nwetara?\", \"choices\": {\"A\": \"20%\", \"B\": \"25%\", \"C\": \"30%\", \"D\": \"15%\"}}, {\"lang\": \"ig\", \"subject\": \"math\", \"answer\": \"C\", \"question\": \"Ọ bụrụ na 2x + 5 = 17, gịnị bụ uru x?\", \"choices\": {\"A\": \"4\", \"B\": \"5\", \"C\": \"6\", \"D\": \"7\"}}, {\"lang\": \"ig\", \"subject\": \"math\", \"answer\": \"B\", \"question\": \"Akụkụ nke triangle bụ x, 2x na 3x. Gịnị bụ uru x?\", \"choices\": {\"A\": \"digirii 20\", \"B\": \"digirii 30\", \"C\": \"digirii 40\", \"D\": \"digirii 60\"}}, {\"lang\": \"ig\", \"subject\": \"science\", \"answer\": \"B\", \"question\": \"Olee nke n'ime ndị a na-abụghị vector quantity?\", \"choices\": {\"A\": \"Ike\", \"B\": \"Mass\", \"C\": \"Ọsọ\", \"D\": \"Ibu\"}}, {\"lang\": \"en\", \"subject\": \"math\", \"answer\": \"B\", \"question\": \"A car travels at 60 km/h. How far will it go in 2.5 hours?\", \"choices\": {\"A\": \"120 km\", \"B\": \"150 km\", \"C\": \"180 km\", \"D\": \"200 km\"}}, {\"lang\": \"yo\", \"subject\": \"math\", \"answer\": \"B\", \"question\": \"Ọkọ̀ ayọ́kẹ́lẹ́ kan ń rìn ní iyára 60 km/h. Báwo ni yóò ṣe jìna ní wákàtí 2.5?\", \"choices\": {\"A\": \"120 km\", \"B\": \"150 km\", \"C\": \"180 km\", \"D\": \"200 km\"}}, {\"lang\": \"ha\", \"subject\": \"math\", \"answer\": \"B\", \"question\": \"Wata mota tana tafiya da gudu 60 km/h. Nawa za ta tafi cikin awanni 2.5?\", \"choices\": {\"A\": \"120 km\", \"B\": \"150 km\", \"C\": \"180 km\", \"D\": \"200 km\"}}, {\"lang\": \"sw\", \"subject\": \"math\", \"answer\": \"B\", \"question\": \"Gari inatembea kwa kasi 60 km/h. Itafikia mbali gani kwa saa 2.5?\", \"choices\": {\"A\": \"120 km\", \"B\": \"150 km\", \"C\": \"180 km\", \"D\": \"200 km\"}}, {\"lang\": \"ig\", \"subject\": \"math\", \"answer\": \"B\", \"question\": \"Otu ụgbọ ala na-eme 60 km/h. Kedu anya ọ ga-eme n'ime awa 2.5?\", \"choices\": {\"A\": \"120 km\", \"B\": \"150 km\", \"C\": \"180 km\", \"D\": \"200 km\"}}]")
print("== WAEC bilingual set (base) ==")
base_waec = eval_waec(model, tok, WAEC_EVAL)
by_lang, overall = waec_summary(base_waec)
print("by language:", by_lang, "| overall:", overall)
with open(f"{OUT}/base_waec.json", "w") as f:
    json.dump({"by_lang": by_lang, "overall": overall, "items": base_waec}, f, indent=2, ensure_ascii=False)

In [ ]:

# Cell 9: BASELINE judge demo - the 2 exact test prompts from metadata.json
DEMO_PROMPTS = [
    ("tp_001 (EN)", "A trader buys a bag of rice for N24,000 and sells it for N30,000. Calculate the percentage profit and show your working step by step."),
    ("tp_002 (YO)", "Oníṣòwò kan ra àpò ìrẹsì kan ní N24,000, ó sì tà á ní N30,000. Ṣe ìṣirò ìpín ọgọ́rùn-ún èrè rẹ̀, kí o sì ṣàlàyé ìgbésẹ̀ kọ̀ọ̀kan."),
]
print("== judge demo (base) ==")
base_demo = {}
for pid, p in DEMO_PROMPTS:
    out = generate_answer(model, tok, chat_prompt(p), max_new=256)
    base_demo[pid] = out
    print(f"\n--- {pid} ---\n{out}")
with open(f"{OUT}/base_demo.json", "w") as f:
    json.dump(base_demo, f, indent=2, ensure_ascii=False)



# =====================================================================
#  FINE-TUNING (QLoRA) - the math and scientific-reasoning experiment
#  Training sources (all verified on Hugging Face):
#   - masakhane/afrimgsm: GSM8k math word problems in 16+ African
#     languages (yor, hau, swa, ibo, eng used here) - core math corpus
#   - masakhane/afrimmlu: MMLU translated into African languages + English
#     (yor/hau/swa/ibo/eng) - broad math + science reasoning, multilingual
#   - cais/mmlu (STEM subjects): deep English math + science coverage
#   - openai/gsm8k: extra English math word problems
#   - worldboss/waec-integrated-science-2007: real WAEC past questions
#   - honourjesus/nllb-hausa-waec-translations: WAEC content in Hausa
#   - curated bilingual step-by-step pairs (embedded in this notebook)
#  Each source is optional: if one fails to download, training continues
#  with the rest (set TRAIN_CAPS below to control runtime on T4).
# =====================================================================


In [ ]:
import json
from datasets import load_dataset, Dataset

# Cell 10: build the training dataset
TRAIN_CAPS = {
    "afrimgsm_yor": 300, "afrimgsm_hau": 300, "afrimgsm_swa": 300,
    "afrimgsm_ibo": 300, "afrimgsm_eng": 150, "waec2007": 29,
    "hausa_waec": 300, "afrimmlu_eng": 150, "afrimmlu_yor": 150,
    "afrimmlu_hau": 150, "afrimmlu_swa": 150, "afrimmlu_ibo": 150,
    "mmlu_stem": 1500, "gsm8k_eng": 400
}

def to_msgs(instruction, response):
    # This function now returns a dictionary with a 'messages' key
    # so that Dataset.from_list can create a dataset with a 'messages' column.
    return {
        "messages": [
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": response}
        ]
    }

all_pairs = []

# Initialize CURATED_TRAIN if not already defined
# Assuming it might be defined elsewhere or is intended to be empty for now
if 'CURATED_TRAIN' not in locals() and 'CURATED_TRAIN' not in globals():
    CURATED_TRAIN = []

# Load curated pairs from variable
for p in CURATED_TRAIN:
    all_pairs.append(to_msgs(p["instruction"], p["response"]))

# ─── 1. afrimgsm ───
for cfg, cap in [
    ("yor", TRAIN_CAPS["afrimgsm_yor"]),
    ("hau", TRAIN_CAPS["afrimgsm_hau"]),
    ("swa", TRAIN_CAPS["afrimgsm_swa"]),
    ("ibo", TRAIN_CAPS["afrimgsm_ibo"]),
    ("eng", TRAIN_CAPS["afrimgsm_eng"])
]:
    try:
        ds = load_dataset("masakhane/afrimgsm", cfg, split="train")
        ds = ds.select(range(min(cap, len(ds))))
        for row in ds:
            all_pairs.append(to_msgs(
                "Solve this mathematics problem step by step.\n\n" + row["question"],
                row["answer"]
            ))
        print(f"afrimgsm [{cfg}]: {len(ds)}")
    except Exception as e:
        print(f"afrimgsm [{cfg}] error: {e}")

# ─── 2. WAEC Integrated Science 2007 ───
try:
    ds = load_dataset("worldboss/waec-integrated-science-2007", split="train")
    ds = ds.select(range(min(TRAIN_CAPS["waec2007"], len(ds))))
    for row in ds:
        all_pairs.append(to_msgs(
            "Answer this WAEC science question.\n\n" + row["Question"],
            row["Answer"]
        ))
    print(f"waec2007: {len(ds)}")
except Exception as e:
    print(f"waec2007 skipped: {e}")

# ─── 3. Hausa WAEC (bypass corrupted metadata, read parquet directly) ───
try:
    from huggingface_hub import hf_hub_download
    import pandas as pd
    from datasets import Dataset

    # Download the raw parquet file directly (bypasses dataset_info.json schema)
    parquet_path = hf_hub_download(
        repo_id="honourjesus/nllb-hausa-waec-translations",
        filename="data/train-00000-of-00001.parquet",
        repo_type="dataset"
    )

    # Read with pandas — uses the ACTUAL parquet schema, not the broken metadata
    df = pd.read_parquet(parquet_path)
    print(f"hausa-waec columns: {list(df.columns)}")
    print(f"hausa-waec shape: {df.shape}")
    print(f"hausa-waec sample:\n{df.head(1).to_dict(orient='records')}")

    # Convert to Hugging Face Dataset
    ds = Dataset.from_pandas(df)

    n = 0
    for row in ds:
        if n >= TRAIN_CAPS["hausa_waec"]:
            break

        # Actual columns from the parquet:
        # question_text_en, correct_answer_en,
        # question_text_ha_pred, correct_answer_ha_pred

        ha_q = row.get("question_text_ha_pred")
        ha_a = row.get("correct_answer_ha_pred")
        en_q = row.get("question_text_en")
        en_a = row.get("correct_answer_en")

        # Skip rows with missing Hausa translations
        if not ha_q or not ha_a:
            continue

        # Option A: Hausa question → Hausa answer (best for Hausa-native tutoring)
        all_pairs.append(to_msgs(
            "Amsa wannan tambayar lissafi a harshen Hausa.\n\n" + str(ha_q),
            str(ha_a)
        ))

        # Option B: English question → Hausa answer (cross-lingual)
        # Uncomment if you want both:
        # if en_q and en_a:
        #     all_pairs.append(to_msgs(
        #         "Answer this WAEC question in Hausa.\n\n" + str(en_q),
        #         str(ha_a)
        #     ))

        n += 1

    print(f"hausa-waec (loaded): {n}")

except Exception as e:
    print(f"hausa-waec error: {e}")
    import traceback
    traceback.print_exc()

# Create the training dataset from all_pairs
train_ds = Dataset.from_list(all_pairs)
print(f"Training dataset created with {len(train_ds)} examples.")

# Define a simple formatting function for SFTTrainer
def format_fn(example):
    # The 'example' is now a dictionary with a 'messages' key
    # containing the chat template structure.
    return example["messages"]

In [ ]:
# Cell 11: QLoRA fine-tune (PEFT + TRL SFTTrainer)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# If re-running this cell, ignore the PEFT warning — it's non-fatal

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

# Calculate warmup steps (~5% of total steps)
total_steps = (len(train_ds) // (2 * 4)) * 3
warmup_steps = max(1, int(total_steps * 0.05))

sft_config = SFTConfig(
    output_dir=f"{OUT}/checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    bf16=False,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to=[],
    seed=SEED,
    max_seq_length=1024,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    formatting_func=format_fn,
)
trainer.train()

# save LoRA adapter
adapter_dir = f"{OUT}/lora_adapter"
model.save_pretrained(adapter_dir)
print("adapter saved to", adapter_dir)

## Post-training (same harness)

In [ ]:
# Cell 12: merge adapter into float16 base for honest post-training eval
# The merged weights are exactly what Cell 15 exports to GGUF, so eval == artifact.
import shutil
del model
gc.collect(); torch.cuda.empty_cache()
from peft import PeftModel
from transformers import AutoModelForCausalLM
base_fp = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base_fp, f"{OUT}/lora_adapter")
model = model.merge_and_unload()
model.eval()
print("merged fine-tuned model loaded (float16, == GGUF)")


In [ ]:

# Cell 13: POST-TRAINING evals (identical harness)
print("== arc_easy (fine-tuned) ==")
ft_arc = eval_arc(model, tok, train_rows, test_rows, n=ARC_N)
print("fine-tuned arc_easy:", ft_arc)
with open(f"{OUT}/ft_arc.json", "w") as f: json.dump(ft_arc, f, indent=2)

print("\n== WAEC bilingual set (fine-tuned) ==")
ft_waec = eval_waec(model, tok, WAEC_EVAL)
by_lang_ft, overall_ft = waec_summary(ft_waec)
print("by language:", by_lang_ft, "| overall:", overall_ft)
with open(f"{OUT}/ft_waec.json", "w") as f:
    json.dump({"by_lang": by_lang_ft, "overall": overall_ft, "items": ft_waec}, f, indent=2, ensure_ascii=False)

print("\n== judge demo (fine-tuned) ==")
ft_demo = {}
for pid, p in DEMO_PROMPTS:
    out = generate_answer(model, tok, chat_prompt(p), max_new=256)
    ft_demo[pid] = out
    print(f"\n--- {pid} ---\n{out}")
with open(f"{OUT}/ft_demo.json", "w") as f:
    json.dump(ft_demo, f, indent=2, ensure_ascii=False)

In [ ]:
# Cell 13b: FT_EFFECTIVE probe - did fine-tuning actually change model behaviour?
# Catches silent failures where fine_tuned == base (training no-op / adapter not applied),
# which a plain accuracy delta would misreport as merely "not improved".
def _probe_arc(m, t, rows, n=ARC_N, shots=ARC_SHOT):
    preds = []
    for row in rows[:n]:
        prompt = build_arc_prompt(row["question"], row["choices"], rows[:shots])
        labels = list(row["choices"].keys())
        ll, _ = letters_loglik(m, t, prompt, labels)
        g = extract_letter(generate_answer(m, t, prompt), labels)
        preds.append((str(ll), str(g), str(row["answer"])))
    return preds

def _probe_waec(m, t, items):
    preds = []
    for it in items:
        prompt = chat_prompt(it["question"], it["choices"])
        g = extract_letter(generate_answer(m, t, prompt), list(it["choices"].keys()))
        preds.append((str(g), str(it["answer"]), it["lang"]))
    return preds

# ft predictions from the already-merged model (== GGUF)
ft_pred_arc = _probe_arc(model, tok, test_rows, n=ARC_N)
ft_pred_waec = _probe_waec(model, tok, WAEC_EVAL)
# fresh base predictions (float16) for an honest comparison
_probe_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
base_pred_arc = _probe_arc(_probe_base, tok, test_rows, n=ARC_N)
base_pred_waec = _probe_waec(_probe_base, tok, WAEC_EVAL)
del _probe_base; gc.collect(); torch.cuda.empty_cache()

arc_diff = sum(1 for b, f in zip(base_pred_arc, ft_pred_arc) if b[0] != f[0] or b[1] != f[1])
waec_diff = sum(1 for b, f in zip(base_pred_waec, ft_pred_waec) if b[0] != f[0])
FT_EFFECTIVE = (arc_diff > 0) or (waec_diff > 0)
print(f"FT_EFFECTIVE: {FT_EFFECTIVE} (arc preds differ={arc_diff}/{ARC_N}, waec preds differ={waec_diff}/{len(WAEC_EVAL)})")
if not FT_EFFECTIVE:
    print("!! WARNING: fine-tuned model produces IDENTICAL predictions to base.")
    print("   The fine-tune had no measurable effect (training no-op / adapter not loaded).")
    print("   Treat arc_improved / waec_improved as INVALID, not merely 'not improved'.")


In [ ]:

# Cell 14: comparison + charts + download
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

def plot_bar(labels, base_vals, ft_vals, title, fname):
    x = np.arange(len(labels)); w = 0.35
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x - w/2, base_vals, w, label="base", color="#94a3b8")
    ax.bar(x + w/2, ft_vals, w, label="fine-tuned", color="#f59e0b")
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylim(0, 1); ax.set_title(title); ax.legend()
    for xi, b, f in zip(x, base_vals, ft_vals):
        ax.text(xi - w/2, b + 0.02, f"{b:.2f}", ha="center", fontsize=8)
        ax.text(xi + w/2, f + 0.02, f"{f:.2f}", ha="center", fontsize=8)
    plt.tight_layout(); plt.savefig(f"{OUT}/{fname}", dpi=150); plt.show()

# 1) arc_easy
plot_bar(["arc_easy (loglik)", "arc_easy (gen)"],
         [base_arc["arc_easy_ll_acc"], base_arc["arc_easy_gen_acc"]],
         [ft_arc["arc_easy_ll_acc"], ft_arc["arc_easy_gen_acc"]],
         "arc_easy: base vs fine-tuned (higher is better)", "chart_arc.png")

# 2) WAEC by language
langs = sorted(set([r["lang"] for r in WAEC_EVAL]))
plot_bar(langs,
         [by_lang.get(l, 0) for l in langs],
         [by_lang_ft.get(l, 0) for l in langs],
         "Reasoning set by language: base vs fine-tuned", "chart_waec_lang.png")

# 3) Reasoning overall + judge demo quality marker
plot_bar(["Reasoning overall"],
         [overall], [overall_ft],
         "Reasoning overall accuracy", "chart_waec_overall.png")

# results bundle
summary = {
    "base": {"arc": base_arc, "waec_by_lang": by_lang, "waec_overall": overall},
    "fine_tuned": {"arc": ft_arc, "waec_by_lang": by_lang_ft, "waec_overall": overall_ft},
    "verdict": {
        "arc_improved": bool(FT_EFFECTIVE) and (ft_arc["arc_easy_ll_acc"] > base_arc["arc_easy_ll_acc"]),
        "waec_improved": bool(FT_EFFECTIVE) and (overall_ft > overall),
        "ft_effective": bool(FT_EFFECTIVE),
    },
}
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(json.dumps(summary["verdict"], indent=2))

# download bundle
import shutil
shutil.make_archive("/content/adtc_results", "zip", OUT)
from google.colab import files
files.download("/content/adtc_results.zip")
print("Done. Keep the zip - it contains summary.json + all raw evals + the LoRA adapter.",
)



In [ ]:
# Cell 15: merge adapter -> GGUF Q4_K_M (LOCAL build, no HF token needed)
# Fix for Protobuf ImportError in some environments
import subprocess
subprocess.run(["pip", "install", "-q", "-U", "protobuf"], check=True)

import os, gc, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

# 1) merge LoRA into the float16 base
print("merging LoRA adapter into base (float16) on CPU to save VRAM...")
if 'model' in locals() or 'model' in globals():
    del model
gc.collect(); torch.cuda.empty_cache()

# Load on CPU for merging to avoid ValueError/Offload errors on T4
base_fp = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="cpu")
adapter = PeftModel.from_pretrained(base_fp, f"{OUT}/lora_adapter")
adapter = adapter.merge_and_unload()
MERGED = "/content/hekima_merged"
adapter.save_pretrained(MERGED); tok.save_pretrained(MERGED)
print("merged model ->", MERGED)

# Free CPU RAM
del base_fp, adapter; gc.collect()

# 2) clone + build llama.cpp, then convert + quantize
print("cloning + building llama.cpp (shallow)...")
subprocess.run(["pip", "install", "-q", "cmake", "gguf"], check=True)
if not os.path.exists("/content/llama.cpp"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp", "/content/llama.cpp"], check=True)
subprocess.run(["pip", "install", "-q", "-r", "/content/llama.cpp/requirements.txt"], check=True)
subprocess.run(["cmake", "-B", "/content/llama.cpp/build", "-DGGML_CUDA=OFF", "/content/llama.cpp"], check=True)
subprocess.run(["cmake", "--build", "/content/llama.cpp/build", "--config", "Release", "-j", "2"], check=True)
GGUF_FP16 = "/content/hekima_fp16.gguf"
subprocess.run(["python", "/content/llama.cpp/convert_hf_to_gguf.py", MERGED, "--outfile", GGUF_FP16], check=True)
GGUF_Q4 = "/content/hekima-tiny-aya-q4_k_m.gguf"
subprocess.run(["/content/llama.cpp/build/bin/llama-quantize", GGUF_FP16, GGUF_Q4, "Q4_K_M"], check=True)
print("quantized GGUF ->", GGUF_Q4, "| size GB:", round(os.path.getsize(GGUF_Q4) / 1e9, 2))

# 3) optional upload to HF (only if HF_TOKEN + HF_REPO are BOTH set)
hf_token = os.environ.get("HF_TOKEN", "")
HF_REPO = os.environ.get("HF_REPO", "")
if hf_token and HF_REPO:
    try:
        from huggingface_hub import HfApi, login
        login(token=hf_token, add_to_git_credential=False)
        api = HfApi()
        api.create_repo(repo_id=HF_REPO, exist_ok=True)
        api.upload_file(path_or_fileobj=GGUF_Q4, path_in_repo=os.path.basename(GGUF_Q4), repo_id=HF_REPO)
        print("uploaded -> https://huggingface.co/" + HF_REPO)
    except Exception as e:
        print("UPLOAD SKIPPED (", type(e).__name__, "):", str(e)[:200])
        print("Local GGUF ready at", GGUF_Q4, "- upload manually later.")
else:
    print("HF_TOKEN/HF_REPO not set - skipped upload. Local GGUF ready at", GGUF_Q4)

print("Next: set HEKIMA_MODEL_URL in download_model.sh and update metadata.json model path to this GGUF.")


## How to read the results

**What this settles:** whether math and scientific-reasoning fine-tuning (AfriGSM + science + bilingual pairs)
actually helps the automated half of S_acc (arc_easy) and the judge half (reasoning-style
bilingual answers).

- If `arc_easy` improves (verdict `arc_improved: true`): math and scientific-reasoning data aligns with the
  benchmark - the reasoning pivot wins on every axis. Strong line for REPORT.md.
- If `arc_easy` stays flat or drops slightly: not fatal - the +15% African Alpha bonus
  and the qualitative judge half (reasoning bilingual accuracy + step-by-step Yoruba answers)
  are the real pitch. A small arc_easy dip is acceptable if reasoning/language scores rise.
- The absolute arc_easy numbers here are from transformers on Colab (float16, T4).
  The official audit runs llama.cpp + Q4_K_M on the 8 GB laptop, so absolute values
  differ - but the DELTA (base vs fine-tuned) is what matters and transfers.

**Artifacts in the zip:** summary.json (verdict), base/ft arc + waec raw evals,
judge demo transcripts (tp_001 EN + tp_002 YO, before/after), the LoRA adapter,
and the 3 comparison charts.

**Next steps after this run:**
1. Cell 15 (below) merges the adapter -> GGUF Q4_K_M and uploads to HF. After it
   finishes, set `HEKIMA_MODEL_URL` in download_model.sh and update `metadata.json`
   (`model.name` + `_runtime.model_path`) to the uploaded Hekima GGUF.
2. Send me summary.json (or the zip) - I interpret it and update REPORT.md, then we
   run the official profiler (llama.cpp + Q4_K_M on the 8 GB laptop) for S_eff.

- `ft_effective` in the verdict tells you whether the fine-tune changed the model at all. If `false`, the run is broken (training no-op / adapter not applied) - do not report `arc_improved` as a real result; re-run training/merge first.


In [ ]:
import json
print(json.load(open("/content/adtc_results/summary.json")))